## Archive SARCOP Incident Data

SARCOP v10 Archive Features is a configuration of ArcGIS Notebooks which can be used by FEMA/NAPSG to archive and backup incident data generated by SARCOP actions.  The notebook will extract SARCOP features from a specific incident based on variables associated with that incident's Incident Area (sublayer 7 of the SARCOP SIFS v10 Master Feature Layer) to copy to an archived layer.  Optionally, those incidents can then be deleted by the notebook so the dashboards only show the current active incidents.

To prepare an incident for archival, ensure that the incident's Incident Area feature contains an Incident Name, a Start Date, and an End Date. If no End Date has been supplied, the script will use the current date/time as the End Date. Incident data is gathered by generating a selection based on the incident feature's footprint and the incident feature's Start and End Dates. All SIFS data found within the incident's footprint and created during the time window of the incident (between the Start and End Dates for the incident) will be transferred to the Archive and tagged with the Incident Name for future reference.

1. Click <b>Cell > Run All</b> to load the Archive SARCOP Incident Data form in this notebook.  Please note this operation may take a moment for the form to appear.  Do not refresh the page.
2. Click <b>Select an Incident to Archive</b>.  Next, click the <b>Select an event</b> dropdown and choose your event.   Click the <b>Archive Incident Features</b> to copy features to the SARCOP Archive service layer.  Please note that incidents with a large number of photos may take several minutes.  Do not refresh the page.
3.  Click the link of the SARCOP Archive layer to open the item in your ArcGIS Organization to confirom all features have been copied.  
4.  <nobr>Optionally, choose <b>Select an Incident to Delete</b> option by selecting an event and click <b>Delete Incident Features.</b> <p style="color:Red;">WARNING: This will delete all features of the chosen incident from the source SARCOP Data layer. </p></nobr>

5.  Click the link of the SARCOP Master layer to open the item in your ArcGIS Organization and verify the features were deleted.


In [ ]:
from datetime import datetime as dt
import shutil
import os
import ipywidgets as widgets
import traceback
import warnings
from IPython.display import display, HTML, clear_output
from ipywidgets import Layout, VBox
from arcgis.gis import GIS
from arcgis.features import FeatureCollection, manage_data
from arcgis.geometry import Geometry
from arcgis.geometry.filters import intersects
import pandas as pd
import requests
from arcgis.features import FeatureLayerCollection
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from zipfile import ZipFile
import arcpy
import shutil

import time

warnings.filterwarnings('ignore')

gis = GIS("home")

# Item ID of SIFS master layer
sifs_item_id = '<MASTER LAYER ID>'

# Item ID of SIFS archive layer
sifs_archive_item_id = '<MASTER ARCHIVE LAYER ID'

# Search Coverage
search_coverage_id = '<SEARCH COVERAGE LAYER ID>' # Point to the production view

search_coverage_archive_id = '<SEARCH COVERAGE ARCHIVE LAYER ID>'


# Structure Hazard Layers

structure_hazard_id = '<STRUCTURE HAZARD LAYER ID>' # Master

structure_hazard_archive = '<STRUCTURE HAZARD LAYER ARCHIVE ID>'

### NOTE: ADD NEW LAYERS HERE ### 


def generate_form(button_text, desc_text, button_tag):
    """ 
    Creates a form displaying an Incident Dropdown menu and a Archive button
    
    Parameters: 
        button_text (str): Text to be centered on Archive button
        desc_text (str): description text above dropdown menu
    Returns
        ipywidgets.VBox: archive form
    
    """
    description = widgets.HTML(
        value="<i>{}</i>".format(desc_text)
    )
    
    se_selector = widgets.Dropdown(
        description = 'Select an event:',
        layout = Layout(width='initial', height='40px'),
        style = {'description_width': 'initial'},
        disabled = False
    )

    deploy_button = widgets.Button(
        description=button_text,
        button_style="primary",
        layout=Layout(width='initial', height='40px'),
        disabled = False
    )
    setattr(deploy_button, "tag", button_tag)

    deploy_form_layout = Layout(display='flex',flex_flow='column', width='700px')
    deploy_form = VBox([description, se_selector, deploy_button], layout=deploy_form_layout)
    return deploy_form

incident_form = widgets.Accordion()
#incident_form.set_title(0, 'Select an Incident to Archive')
#incident_form.set_title(1, 'Select an Incident to Delete')
#incident_form.set_title(2, 'Select an Incident to Archive (Public Assistance)')
#incident_form.set_title(3, 'Select an Incident to Delete (Public Assistance)')

fema_checkbox = widgets.Checkbox(
    value=False,
    description='Was FEMA Deployed?',
    disabled=False,
    indent=False
)
display(fema_checkbox)

gdb_checkbox = widgets.Checkbox(
    value=True,
    description='Create Geodatabase Backup?',
    disabled=False,
    indent=False
)
display(gdb_checkbox)
def reset_form(archive_form):
    """ 
    Resets archive form to remove progress bar and output from previous run
    
    """
    archive_form.children = archive_form.children[:3]
    
def generate_progressor(steps):
    """ 
    Uses ipywidgets to generate a progress bar with custom progress messages and styling
    to report progress of deployment

    Parameters: None

    Returns: 
        ipywidgets.VBox: progress bar
    """
    progressor_label = widgets.HTML(style = {'description_width': 'initial'})
    progressor = widgets.IntProgress(min=0, max=steps, step=1, layout=Layout(width='initial'))
    progressor.style.bar_color = 'gray'
    return widgets.VBox([progressor_label, progressor], layout=Layout(width='690px', border='1px solid silver', margin='5px', padding='5px'))
    
def update_progressor(prg_box, msg=None, error_state=False, progress=True):
    """ 
    Updates the message and bar length of progress bar generated
    by the generate_progressor function. If in error state the bar
    is changed to red and progress is stopped.

    Parameters: 
        prg_box (ipywidgets.VBox): Progress bar widget
        msg (str): The message to show on the progress bar lable
        error_state (bool): The current error status of the deployment
        
    """
    progressor_label = prg_box.children[0]
    progressor = prg_box.children[1]
    if error_state:
        progressor.style.bar_color = 'red'
        prg_box.layout.border = '1px solid red'
        progressor_label.value = progressor_label.value.replace('gray','red')
    else:
        label_format = '<p style="color:gray;">{}</p>'
        if progress:
            progressor.value += 1
        if msg:
            progressor_label.value = label_format.format(msg)

# Code hiding logic from here: https://stackoverflow.com/a/53136940
# Hides code cell in notebook after selecting "Cell > Run All"
toggle_code_javascript_functions = {False: "hide()", True: "show()"}
toggle_code_button_descriptions  = {False: "Show code", True: "Hide code"}
toggle_code_button_icons = {False: "plus", True: "minus"}

def toggle_code(state):
    """
    Toggles the JavaScript show()/hide() function on the div.input element.
    """
    output_string = "<script>$(\"div.input\").{}</script>"
    output_args   = (toggle_code_javascript_functions[state],)
    output        = output_string.format(*output_args)

    display(HTML(output))

def toggle_code_button_action(value):
    """
    Calls the toggle_code function and updates the button description.
    """
    state = value.new

    toggle_code(state)

    value.owner.description = toggle_code_button_descriptions[state]
    value.owner.icon = toggle_code_button_icons[state]

toggle_code_state = False
toggle_code(toggle_code_state)
toggle_code_button = widgets.ToggleButton(toggle_code_state, description = toggle_code_button_descriptions[toggle_code_state], icon=toggle_code_button_icons[toggle_code_state])
toggle_code_button.observe(toggle_code_button_action, "value")

def get_related_records_oids(related_record_query_result, comments_table_objectid_field):
    """
    Creates a dictionary with key value pairs of Related Record ObjectID : Parent ObjectID
    
    Parameters:
        related_record_query_result (dict): Result of parent_layer.query_related_records()
        comments_table_objectid_field (str): The name of the ObjectID field on the comments table
        
    Returns:
        Dictionary: {'Comment Object ID' : 'Parent Object ID',...n}
    
    """    
    parent_lookup_dict = {}
    for group in related_record_query_result['relatedRecordGroups']:
        for record in group['relatedRecords']:
            record_oid = record['attributes'][comments_table_objectid_field]
            parent_record_oid = group['objectId']
            parent_lookup_dict[str(record_oid)] = parent_record_oid
    
    return parent_lookup_dict

def migrate_attachments(original_table, archive_table, ids, results_lookup_dict, download_folder):
    """
    Downloads attachments from a table or feature layer and uploads those attachments to another
    table or feature layer based on a shared globalid between the two layers.
    
    Parameters:
        original_table (arcgis.features.FeatureLayer or arcgis.features.Table): Original feature layer or related table
        archive_table (arcgis.features.FeatureLayer or arcgis.features.Table): Archive feature layer or related table
        ids (List(int)): List of ObjectIDs to search for associated attachments
        results_lookup_dict (dict): GlobalID : ObjectID dict from edit_features() results
        download_folder (str): Folder to download attachments
    
    """ 
    
    original_object_id_field = original_table.manager.properties["objectIdField"]
    original_global_id_field = original_table.manager.properties["globalIdField"]
    
    oids_list_string = ",".join([str(oid) for oid in ids])
    attachment_infos = original_table.attachments.search(object_ids = oids_list_string)

    if attachment_infos:
        if gis.properties.isPortal:
            # Patch for enterprise because table.attachments.search() sets 'PARENTGLOBALID' to 'N/A'
            # we use a standard table query (table.query()) to get parent global ids instead
            results = original_table.query(object_ids=oids_list_string,out_fields=original_global_id_field).features
            attachments_oid_lookup = {str(feat.get_value(original_object_id_field)):feat.get_value(original_global_id_field) for feat in results}
        else:
            attachments_oid_lookup = {str(att['PARENTOBJECTID']):att['PARENTGLOBALID'] for att in attachment_infos}
        downloads = original_table.attachments.download(oid=list(attachments_oid_lookup.keys()), save_path=download_folder)

        for att in downloads:
            object_id = att.split(os.sep)[-3]
            global_id = attachments_oid_lookup[object_id]

            corresponding_archive_oid = results_lookup_dict[global_id]
            archive_table.attachments.add(corresponding_archive_oid,att)

        shutil.rmtree(download_folder)
    
def identify_features_to_archive(original_table, archive_table, incident_id="", object_ids=""):
    """
    Compares an archive layer to an original layer and determines which features have not yet been
    copied to the archive layer.
    
    Parameters:
        original_table (arcgis.features.FeatureLayer or arcgis.features.Table): Original feature layer or related table
        archive_table (arcgis.features.FeatureLayer or arcgis.features.Table): Archive feature layer or related table
        incident_id (str): The ID of the incident
        object_ids(str): Comma separated list of ObjectIDs
        
    Returns:
        features_to_archive: List of arcgis.features.Feature
        feature_ids: ObjectIDs of the features to archive
    
    """ 
    original_global_id_field = original_table.manager.properties["globalIdField"]
    archive_global_id_field = archive_table.manager.properties["globalIdField"]
    
    original_object_id_field = original_table.manager.properties["objectIdField"]
    archive_object_id_field = archive_table.manager.properties["objectIdField"]
    
    if incident_id:
        try:
            where_clause = "event_name = '{}'".format(incident_id)
            original_features = original_table.query(where=where_clause).features
        except:
            where_clause = "IncidentName = '{}'".format(incident_id)
            original_features = original_table.query(where=where_clause).features
    else:
        where_clause = "1=1"
        original_features = original_table.query(object_ids=object_ids).features
        
    archive_features = archive_table.query(where=where_clause, out_fields=archive_global_id_field).features
    archive_feature_global_ids = {f.get_value(archive_global_id_field) for f in archive_features}
    
    features_to_archive = [f for f in original_features if f.get_value(original_global_id_field) not in archive_feature_global_ids]
    feature_ids = [f.get_value(original_object_id_field) for f in features_to_archive]
    
    return features_to_archive, feature_ids

def identify_incident_data(original_table, archive_table, incident_id, where_clause, incident_filter):
    """
    Compares an archive layer to an original layer and determines which features have not yet been
    copied to the archive layer.
    
    Parameters:
        original_table (arcgis.features.FeatureLayer or arcgis.features.Table): Original feature layer or related table
        archive_table (arcgis.features.FeatureLayer or arcgis.features.Table): Archive feature layer or related table
        incident_id (str): The ID of the incident - generated by the Incident Dropdown Select menu
        where_clause (str): A SQL expression to select features based on a supplied date range
        incident_filter (dict): An arcgis.Geometry.Filters dictionary to perform a selection by intersection with the supplied Incident geometry
        
    Returns:
        features_to_archive: List of arcgis.features.Feature
        feature_ids: ObjectIDs of the features to archive
    
    """
    original_global_id_field = original_table.manager.properties["globalIdField"]
    archive_global_id_field = archive_table.manager.properties["globalIdField"]
    
    original_object_id_field = original_table.manager.properties["objectIdField"]
    archive_object_id_field = archive_table.manager.properties["objectIdField"]

    original_features = original_table.query(where=where_clause, geometry_filter=incident_filter).features

    try:
        inc_where_clause = "event_name = '{}'".format(incident_id)
        archive_features = archive_table.query(where=inc_where_clause).features
    except:
        inc_where_clause = "IncidentName = '{}'".format(incident_id)
        archive_features = archive_table.query(where=inc_where_clause).features
    archive_feature_global_ids = {f.get_value(archive_global_id_field) for f in archive_features}
    features_to_archive = [f for f in original_features if f.get_value(original_global_id_field) not in archive_feature_global_ids]
    feature_ids = [f.get_value(original_object_id_field) for f in features_to_archive]

    return features_to_archive, feature_ids

def send_edits_to_service(feature_set, feature_layer, incident_id=""):
    """
    Sends edits to a feature layer or table in batches of 100
    
    Parameters:
        feature_set(List of arcgis.features.feature): List of features to add
        feature_layer(arcgis.features.FeatureLayer): feature layer to edit
        
    Returns:
        all_results (Dict): 'addResults' dict
    
    """ 
    
    num_features = len(feature_set)

    if num_features > 100:
        chunk = 100
    else:
        chunk = num_features

    processed_features = 0
    all_results = []
    while processed_features < num_features:
        next = processed_features + chunk
        features_chunk = feature_set[processed_features:next]
        for feature in features_chunk:
            if incident_id:
                feature.attributes['event_name'] = incident_id
                #feature.attributes['IncidentName'] = incident_id # Don't see this in v10 - CR
        #feature.attributes['team_user'] = '' # Don't see this in v10 - CR
        feature.attributes['email'] = ''

        results = feature_layer.edit_features(adds=features_chunk, use_global_ids=True)
        all_results.extend(results['addResults'])

        processed_features += chunk
    
    return all_results

def refresh_form(new_form=False, selected_index=0):
    """
    Refreshes the user interface for the notebook. Populates the Incident Dropdowns with new incidents
    
    Parameters:
        new_form (bool) = Determines whether to clear the UI and rebuild or leave whats already there after an archive process
        selected_index (Int) = selects which menu to be open on refresh
    
    """    
    if new_form:
        #
        archive_instructions = ("This will copy all SAR features and related records to the archive feature service. This will only show incidents that are active.")
        archive_form = generate_form("Archive Incident Features", archive_instructions, "ind")
        delete_instructions = ("This will delete all SAR features and related comments from the original layer."
                              " Prior to deleting please verify that features have been copied to the archive layer")
        delete_form = generate_form("Delete Incident Features", delete_instructions, "ind")
        
    else:
        archive_form = incident_form.children[0]
        delete_form = incident_form.children[1]
        
        #pub_archive_form = incident_form.children[2]
        #pub_delete_form = incident_form.children[3]
    filterLayer = gis.content.get('<MOBILE VIEW LAYER ID>')
    filterIncidentArea = filterLayer.layers[7]
    
    sifs_item = gis.content.get(sifs_item_id)
    # SIFS sublayer definitions
    sifs_waypoints = sifs_item.layers[0]
    sifs_worksites = sifs_item.layers[1]
    sifs_points = sifs_item.layers[2]
    sifs_tracklog = sifs_item.layers[3]
    sifs_segments = sifs_item.layers[4]
    sifs_divisions = sifs_item.layers[5]
    sifs_branches = sifs_item.layers[6]
    sifs_incident_areas = sifs_item.layers[7]
    # search coverage
    search_coverage_item = gis.content.get(search_coverage_id)
    search_coverage_points = search_coverage_item.layers[0]

    
    #Structure Hazard
    structure_hazard_item = gis.content.get(structure_hazard_id)
    structure_hazard_eval_points = structure_hazard_item.layers[0]
    structure_risk_plan = structure_hazard_item.layers[1]
    structure_hazard_monitoring = structure_hazard_item.layers[2]
    structure_worksite_axis = structure_hazard_item.layers[3]
    structure_grid = structure_hazard_item.layers[4]
    structure_hazard_risk = structure_hazard_item.layers[5]
    ### NOTE: ADD NEW LAYERS HERE ###
    
    unique_incidents = sorted(list(set([str(f.get_value('event_name')) for f in filterIncidentArea.query(out_fields='event_name').features])))
    if 'None' in unique_incidents:
        unique_incidents.remove('None')
    unique_incidents.remove('Training Only')
        
    if unique_incidents:
        archive_form.children[1].options = unique_incidents
        archive_form.children[1].disabled = False
        archive_form.children[2].disabled = False
        delete_form.children[1].options = unique_incidents
        delete_form.children[1].disabled = False
        delete_form.children[2].disabled = False
    else:
        archive_form.children[1].options = [["No incidents available",""]]
        archive_form.children[1].disabled = True
        archive_form.children[2].disabled = True
        delete_form.children[1].options = [["No incidents available",""]]
        delete_form.children[1].disabled = True
        delete_form.children[2].disabled = True

    incident_form.children=[archive_form, delete_form]
    incident_form.set_title(0, 'Select an Incident to Archive')
    incident_form.set_title(1, 'Select an Incident to Delete')
    incident_form.selected_index = selected_index
    
def delete_archived_features(_):
    """
    This is the main function of the notebook that runs when the delete button is clicked. It identifies features and related records
    that belong to the incident_id selected in the dropdown menu and deletes those features from the original layer.
    
    """

    delete_form = incident_form.children[1]
    progressor_size = 11
    finish_index = 1 
        
    incident_selector = delete_form.children[1]
    
    delete_button = delete_form.children[2]
    delete_button.disabled = True
    
    try:
        # Setup function variables
        reset_form(delete_form)
        incident_id = incident_selector.value
        
        progressor_box = generate_progressor(progressor_size)
        out = widgets.Output()
        delete_form.children += (progressor_box,)
        delete_form.children += (out,)
        
        # Process Features ****************************************************************
        update_progressor(progressor_box, "Getting SARCOP layer information...")
        
        sifs_item = gis.content.get(sifs_item_id)
        # SIFS sublayer definitions
        sifs_waypoints = sifs_item.layers[0]
        sifs_worksites = sifs_item.layers[1]
        sifs_points = sifs_item.layers[2]
        sifs_tracklog = sifs_item.layers[3]
        sifs_segments = sifs_item.layers[4]
        sifs_divisions = sifs_item.layers[5]
        sifs_branches = sifs_item.layers[6]
        sifs_incident_areas = sifs_item.layers[7]
        # Search Coverage
        search_coverage_item = gis.content.get(search_coverage_id)
        search_coverage_points = search_coverage_item.layers[0]
        
        #structure hazard layers
        structure_hazard_item = gis.content.get(structure_hazard_id)
        structure_hazard_eval_points = structure_hazard_item.layers[0]
        structure_risk_plan = structure_hazard_item.layers[1]
        structure_hazard_monitoring = structure_hazard_item.layers[2]
        structure_worksite_axis = structure_hazard_item.layers[3]
        structure_grid = structure_hazard_item.layers[4]
        structure_hazard_risk = structure_hazard_item.layers[5]
        ### NOTE: ADD NEW LAYERS HERE ###
        
        sifs_waypoints_objectid_field = sifs_waypoints.manager.properties["objectIdField"]
        sifs_worksites_objectid_field = sifs_worksites.manager.properties["objectIdField"]
        sifs_points_objectid_field = sifs_points.manager.properties["objectIdField"]
        sifs_tracklog_objectid_field = sifs_tracklog.manager.properties["objectIdField"]
        sifs_segments_objectid_field = sifs_segments.manager.properties["objectIdField"]
        sifs_divisions_objectid_field = sifs_divisions.manager.properties["objectIdField"]
        sifs_branches_objectid_field = sifs_branches.manager.properties["objectIdField"]
        sifs_incident_areas_objectid_field = sifs_incident_areas.manager.properties["objectIdField"]
        # Search Coverage
        search_coverage_objectid_field = search_coverage_points.manager.properties["objectIdField"]

        #Structure Hazard Layers
        structure_hazard_eval_points_objectid_field = structure_hazard_eval_points.manager.properties["objectIdField"]
        structure_risk_plan_objectid_field = structure_risk_plan.manager.properties["objectIdField"]
        structure_hazard_monitoring_objectid_field = structure_hazard_monitoring.manager.properties["objectIdField"]
        structure_worksite_axis_objectid_field = structure_worksite_axis.manager.properties["objectIdField"]
        structure_grid_objectid_field = structure_grid.manager.properties["objectIdField"]
        structure_hazard_risk_objectid_field = structure_hazard_risk.manager.properties["objectIdField"]
        ### NOTE: ADD NEW LAYERS HERE ###
        
        
        sifs_archive_item = gis.content.get(sifs_archive_item_id)
        sifs_archive_waypoints = sifs_archive_item.layers[0]
        sifs_archive_worksites = sifs_archive_item.layers[1]
        sifs_archive_points = sifs_archive_item.layers[2]
        sifs_archive_tracklog = sifs_archive_item.layers[3]
        sifs_archive_segments = sifs_archive_item.layers[4]
        sifs_archive_divisions = sifs_archive_item.layers[5]
        sifs_archive_branches = sifs_archive_item.layers[6]
        sifs_archive_incident_areas = sifs_archive_item.layers[7]
        # Search Coverage
        search_coverage_archive_item = gis.content.get(search_coverage_archive_id)
        search_coverage_archive_points = search_coverage_archive_item.layers[0]

        # Structure Hazard
        structures_archive_item = gis.content.get(structure_hazard_archive)

        structure_hazard_eval_archive = structures_archive_item.layers[0]
        structure_risk_plan_archive = structures_archive_item.layers[1]
        structure_hazard_monitoring_archive = structures_archive_item.layers[2]
        structure_worksite_axis_archive = structures_archive_item.layers[3]
        structure_grid_archive = structures_archive_item.layers[4]
        structure_hazard_risk_archive = structures_archive_item.layers[5]

        ### NOTE: ADD NEW LAYERS HERE ###
        
        update_progressor(progressor_box, "Identifying features to delete from incident: {}...".format(incident_id))
        
        # Get incident area geometry and start/end dates
        event_name_where_clause = "event_name = '{}'".format(incident_id)
        all_incident_areas = sifs_incident_areas.query(where=event_name_where_clause)
        incident_sr = sifs_incident_areas.properties.extent['spatialReference']
        incident_geom_dict = all_incident_areas.features[0].geometry
        incident_geom_dict['spatialReference'] = incident_sr
        incident_geom = Geometry(incident_geom_dict)
        incident_filter = intersects(incident_geom, sr=incident_sr)
        inc_start = all_incident_areas.features[0].attributes['incident_start_date']
        inc_end = all_incident_areas.features[0].attributes['incident_end_date']
        inc_start_date = dt.fromtimestamp(inc_start/1000).strftime("%m/%d/%Y 00:00:00")
        if inc_end:
            inc_end_date = dt.fromtimestamp(inc_end/1000).strftime("%m/%d/%Y 23:59:59")
        else:
            inc_end_date = dt.now().strftime("%m/%d/%Y 23:59:59")
        
        where_clause = "CreationDate > '{}' AND CreationDate < '{}' AND delete_this <> 'yes_delete'".format(inc_start_date, inc_end_date)
        search_coverage_where = "CreationDate > '{}' AND CreationDate < '{}' AND analysis_type <> 'Staging Run'".format(inc_start_date, inc_end_date)

        # Get all the features that match the incident geometry and time
        all_incident_waypoints = sifs_waypoints.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_worksites = sifs_worksites.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_tracklogs = sifs_tracklog.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_segments = sifs_segments.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_divisions = sifs_divisions.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_branches = sifs_branches.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_points = sifs_points.query(where=where_clause, geometry_filter=incident_filter)
        # search coverage
        all_incident_search_coverage_points = search_coverage_points.query(geometry_filter=incident_filter)

        # Structure hazard layers
        all_incident_structure_hazard_eval_points = structure_hazard_eval_points.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_structure_risk_plan = structure_risk_plan.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_structure_hazard_monitoring = structure_hazard_monitoring.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_structure_worksite_axis = structure_worksite_axis.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_structure_grid = structure_grid.query(where=where_clause, geometry_filter=incident_filter)
        all_incident_structure_hazard_risk = structure_hazard_risk.query(where=where_clause, geometry_filter=incident_filter)
        ### NOTE: ADD NEW LAYERS HERE ###
        
        #Identify features from incident that have not yet been moved to the archive feature service
        new_incident_areas_to_archive, new_incident_areas_ids = identify_features_to_archive(sifs_incident_areas,
                                                                                 sifs_archive_incident_areas,
                                                                                 incident_id=incident_id)
        
        new_waypoints_to_archive, new_waypoints_ids = identify_incident_data(sifs_waypoints,
                                                                                    sifs_archive_waypoints,
                                                                                    incident_id=incident_id,
                                                                                    where_clause=where_clause,
                                                                                    incident_filter=incident_filter)
        new_worksites_to_archive, new_worksites_ids = identify_incident_data(sifs_worksites,
                                                                                sifs_archive_worksites,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_points_to_archive, new_points_ids = identify_incident_data(sifs_points,
                                                                                sifs_archive_points,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_tracklog_to_archive, new_tracklogs_ids = identify_incident_data(sifs_tracklog,
                                                                                sifs_archive_tracklog,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_segments_to_archive, new_segments_ids = identify_incident_data(sifs_segments,
                                                                                sifs_archive_segments,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_divisions_to_archive, new_divisions_ids = identify_incident_data(sifs_divisions,
                                                                                sifs_archive_divisions,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_branches_to_archive, new_branches_ids = identify_incident_data(sifs_branches,
                                                                                sifs_archive_branches,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        # Search Coverage
        new_search_coverage_to_archive, new_search_coverage_ids = identify_incident_data(search_coverage_points,
                                                                                search_coverage_archive_points,
                                                                                incident_id=incident_id,
                                                                                where_clause=search_coverage_where,
                                                                                incident_filter=incident_filter) 
        # Structure Hazard layers
        new_structure_hazard_eval_points_to_archive, new_structure_hazard_eval_points_ids = identify_incident_data(structure_hazard_eval_points,
                                                                                structure_hazard_eval_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        
        new_structure_risk_plan_to_archive, new_structure_risk_plan_ids = identify_incident_data(structure_risk_plan,
                                                                                structure_risk_plan_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        
        new_structure_hazard_monitoring_to_archive, new_structure_hazard_monitoring_ids = identify_incident_data(structure_hazard_monitoring,
                                                                                structure_hazard_monitoring_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        
        new_structure_worksite_axis_to_archive, new_structure_worksite_axis_ids = identify_incident_data(structure_worksite_axis,
                                                                                structure_worksite_axis_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        
        new_structure_grid_to_archive, new_structure_grid_ids = identify_incident_data(structure_grid,
                                                                                structure_grid_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)

        new_structure_hazard_risk_to_archive, new_structure_hazard_risk_ids = identify_incident_data(structure_hazard_risk,
                                                                                structure_hazard_risk_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        ### NOTE: ADD NEW LAYERS HERE ###
    
        if (new_waypoints_ids or new_worksites_ids or new_points_ids or new_tracklogs_ids or new_segments_ids or new_divisions_ids or new_branches_ids or new_incident_areas_ids): # ADD search coverage to this list once archive is ready.
            raise ValueError("Feature(s) identified in original layer that have not yet been archived.\n Please archive these features before deleting.")
        
        waypoints_oid_list = [f.get_value(sifs_waypoints_objectid_field) for f in all_incident_waypoints.features]
        waypoints_oid_list_string = ",".join([str(oid) for oid in waypoints_oid_list])
        worksites_oid_list = [f.get_value(sifs_worksites_objectid_field) for f in all_incident_worksites.features]
        worksites_oid_list_string = ",".join([str(oid) for oid in worksites_oid_list])
        points_oid_list = [f.get_value(sifs_points_objectid_field) for f in all_incident_points.features]
        points_oid_list_string = ",".join([str(oid) for oid in points_oid_list])
        tracklog_oid_list = [f.get_value(sifs_tracklog_objectid_field) for f in all_incident_tracklogs.features]
        tracklog_oid_list_string = ",".join([str(oid) for oid in tracklog_oid_list])
        segments_oid_list = [f.get_value(sifs_segments_objectid_field) for f in all_incident_segments.features]
        segments_oid_list_string = ",".join([str(oid) for oid in segments_oid_list])
        divisions_oid_list = [f.get_value(sifs_divisions_objectid_field) for f in all_incident_divisions.features]
        divisions_oid_list_string = ",".join([str(oid) for oid in divisions_oid_list])
        branches_oid_list = [f.get_value(sifs_branches_objectid_field) for f in all_incident_branches.features]
        branches_oid_list_string = ",".join([str(oid) for oid in branches_oid_list])
        incident_areas_oid_list = [f.get_value(sifs_incident_areas_objectid_field) for f in all_incident_areas.features]
        incident_areas_oid_list_string = ",".join([str(oid) for oid in incident_areas_oid_list])
        # search coverage
        search_coverage_oid_list = [f.get_value(search_coverage_objectid_field) for f in all_incident_search_coverage_points.features]
        search_coverage_oid_list_string = ",".join([str(oid) for oid in search_coverage_oid_list])

        
        # Structure Hazard layers
        structure_hazard_eval_points_oid_list = [f.get_value(structure_hazard_eval_points_objectid_field) for f in all_incident_structure_hazard_eval_points.features]
        structure_hazard_eval_points_oid_list_string = ",".join([str(oid) for oid in structure_hazard_eval_points_oid_list])
        
        structure_risk_plan_oid_list = [f.get_value(structure_risk_plan_objectid_field) for f in all_incident_structure_risk_plan.features]
        structure_risk_plan_oid_list_string = ",".join([str(oid) for oid in structure_risk_plan_oid_list])
        
        structure_hazard_monitoring_oid_list = [f.get_value(structure_hazard_monitoring_objectid_field) for f in all_incident_structure_hazard_monitoring.features]
        structure_hazard_monitoring_oid_list_string = ",".join([str(oid) for oid in structure_hazard_monitoring_oid_list])
        
        structure_worksite_axis_oid_list = [f.get_value(structure_worksite_axis_objectid_field) for f in all_incident_structure_worksite_axis.features]
        structure_worksite_axis_oid_list_string = ",".join([str(oid) for oid in structure_worksite_axis_oid_list])
        
        structure_grid_oid_list = [f.get_value(structure_grid_objectid_field) for f in all_incident_structure_grid.features]
        structure_grid_oid_list_string = ",".join([str(oid) for oid in structure_grid_oid_list])
        
        structure_hazard_risk_oid_list = [f.get_value(structure_hazard_risk_objectid_field) for f in all_incident_structure_hazard_risk.features]
        structure_hazard_risk_oid_list_string = ",".join([str(oid) for oid in structure_hazard_risk_oid_list])
        

        
        ### NOTE: ADD NEW LAYERS HERE ###
        
        if waypoints_oid_list:
            update_progressor(progressor_box, "Deleting waypoints from incident: {}...".format(incident_id))
            sifs_waypoints.delete_features(deletes=waypoints_oid_list_string)
        else:
            update_progressor(progressor_box, "No waypoints to delete...")
        if worksites_oid_list:
            update_progressor(progressor_box, "Deleting worksites from incident: {}...".format(incident_id))
            sifs_worksites.delete_features(deletes=worksites_oid_list_string)
        else:
            update_progressor(progressor_box, "No worksites to delete...")
        if points_oid_list:
            update_progressor(progressor_box, "Deleting event points from incident: {}...".format(incident_id))
            sifs_points.delete_features(deletes=points_oid_list_string)
        else:
            update_progressor(progressor_box, "No event points to delete...")
        if tracklog_oid_list:
            update_progressor(progressor_box, "Deleting tracklogs from incident: {}...".format(incident_id))
            sifs_tracklog.delete_features(deletes=tracklog_oid_list_string)
        else:
            update_progressor(progressor_box, "No tracklogs to delete...")
        if segments_oid_list:
            update_progressor(progressor_box, "Deleting search segments from incident: {}".format(incident_id))
            sifs_segments.delete_features(deletes=segments_oid_list_string)
        else:
            update_progressor(progressor_box, "No search segments to delete...")
        if divisions_oid_list:
            update_progressor(progressor_box, "Deleting divisions from incident: {}".format(incident_id))
            sifs_divisions.delete_features(deletes=divisions_oid_list_string)
        else:
            update_progressor(progressor_box, "No divisions to delete...")
        if branches_oid_list:
            update_progressor(progressor_box, "Deleting branches from incident: {}".format(incident_id))
            sifs_branches.delete_features(deletes=branches_oid_list_string)
        else:
            update_progressor(progressor_box, "No branches to delete...")
        if incident_areas_oid_list:
            update_progressor(progressor_box, "Deleting incident areas from incident: {}...".format(incident_id))
            sifs_incident_areas.delete_features(deletes=incident_areas_oid_list_string)
        else:
            update_progressor(progressor_box, "No incident areas to delete...")
        # Search Coverage
        if search_coverage_oid_list:
            update_progressor(progressor_box, "Deleting search coverage points from incident: {}...".format(incident_id))
            search_coverage_points.delete_features(deletes=search_coverage_oid_list_string)
        else:
            update_progressor(progressor_box, "No search coverage points to delete...")


        
        # Structure Hazard
        if structure_hazard_eval_points_oid_list:
            update_progressor(progressor_box, "Deleting structure hazard evaluation points from incident: {}...".format(incident_id))
            structure_hazard_eval_points.delete_features(deletes=structure_hazard_eval_points_oid_list_string)
        else:
            update_progressor(progressor_box, "No structure hazard evaluation points...")

        if structure_risk_plan_oid_list:
            update_progressor(progressor_box, "Deleting structure risk plan data from incident: {}...".format(incident_id))
            structure_risk_plan.delete_features(deletes=structure_risk_plan_oid_list_string)
        else:
            update_progressor(progressor_box, "No structure risk plan points...")
        
        if structure_hazard_monitoring_oid_list:
            update_progressor(progressor_box, "Deleting structure hazard moniroting data from incident: {}...".format(incident_id))
            structure_hazard_monitoring.delete_features(deletes=structure_hazard_monitoring_oid_list_string)
        else:
            update_progressor(progressor_box, "No structure hazard moniroting data...")

        if structure_worksite_axis_oid_list:
            update_progressor(progressor_box, "Deleting structure worksite axis data from incident: {}...".format(incident_id))
            structure_worksite_axis.delete_features(deletes=structure_worksite_axis_oid_list_string)
        else:
            update_progressor(progressor_box, "No structure worksite axis data...")

        if structure_grid_oid_list:
            update_progressor(progressor_box, "Deleting structure grid data from incident: {}...".format(incident_id))
            structure_grid.delete_features(deletes=structure_grid_oid_list_string)
        else:
            update_progressor(progressor_box, "No structure grid data...")

        if structure_hazard_risk_oid_list:
            update_progressor(progressor_box, "Deleting structure hazard risk data from incident: {}...".format(incident_id))
            structure_hazard_risk.delete_features(deletes=structure_hazard_risk_oid_list_string)
        else:
            update_progressor(progressor_box, "No structure hazard risk data...")

        
            
        ### NOTE: ADD NEW LAYERS HERE ###
        
        update_progressor(progressor_box, "Incident: {} features or records successfully deleted ...".format(incident_id))
        
        out_item = widgets.Output()
        out_item.append_display_data(sifs_item)

        delete_form.children += (out_item,)
        
        refresh_form(selected_index=finish_index)
        
    except Exception as e:
        update_progressor(progressor_box, error_state=True)
        tb = traceback.format_exc()
        out.append_stderr("Delete Unsuccessful. Please try again.\n")
        out.append_stderr("Error:\n{}\n".format(str(e)))
        out.append_stderr(tb)
    finally:
        delete_button.disabled = False
        
def archive_features(_):
    """
    This is the main function of the notebook that runs when the Archive button is clicked. It identifies features and related records
    that belong to the incident_id selected in the dropdown menu and copies those features to the archive feature service.
    
    """
    
    # 'ind' : Individual Assistance
    # 'pub' : Public Assistance
    archive_form = incident_form.children[0]
    progressor_size = 10
    has_comments = True
    finish_index = 0
    
    incident_selector = archive_form.children[1]
    
    archive_button = archive_form.children[2]
    archive_button.disabled = True
    
    try:
        # Setup function variables
        reset_form(archive_form)
        incident_id = incident_selector.value
        
        progressor_box = generate_progressor(progressor_size)
        out = widgets.Output()
        archive_form.children += (progressor_box,)
        archive_form.children += (out,)
        
        # Processing Assessment Features *************************************
        
        update_progressor(progressor_box, "Getting SARCOP layer information...")

        sifs_item = gis.content.get(sifs_item_id)
        # SIFS sublayer definitions
        sifs_waypoints = sifs_item.layers[0]
        sifs_worksites = sifs_item.layers[1]
        sifs_points = sifs_item.layers[2]
        sifs_tracklog = sifs_item.layers[3]
        sifs_segments = sifs_item.layers[4]
        sifs_divisions = sifs_item.layers[5]
        sifs_branches = sifs_item.layers[6]
        sifs_incident_areas = sifs_item.layers[7]
        # Search Coverage
        search_coverage_item = gis.content.get(search_coverage_id)
        search_coverage_points = search_coverage_item.layers[0]

        # Structure Layers
        structure_hazard_item = gis.content.get(structure_hazard_id)
        structure_hazard_eval_points = structure_hazard_item.layers[0]
        structure_risk_plan = structure_hazard_item.layers[1]
        structure_hazard_monitoring = structure_hazard_item.layers[2]
        structure_worksite_axis = structure_hazard_item.layers[3]
        structure_grid = structure_hazard_item.layers[4]
        structure_hazard_risk = structure_hazard_item.layers[5]
        ### NOTE: ADD NEW LAYERS HERE ###

        
        sifs_waypoints_objectid_field = sifs_waypoints.manager.properties["objectIdField"]
        sifs_worksites_objectid_field = sifs_worksites.manager.properties["objectIdField"]
        sifs_points_objectid_field = sifs_points.manager.properties["objectIdField"]
        sifs_tracklog_objectid_field = sifs_tracklog.manager.properties["objectIdField"]
        sifs_segments_objectid_field = sifs_segments.manager.properties["objectIdField"]
        sifs_divisions_objectid_field = sifs_divisions.manager.properties["objectIdField"]
        sifs_branches_objectid_field = sifs_branches.manager.properties["objectIdField"]
        sifs_incident_areas_objectid_field = sifs_incident_areas.manager.properties["objectIdField"]
        # Search Coverage
        search_coverage_objectid_field = search_coverage_points.manager.properties["objectIdField"]

        # Structure Layers
        structure_hazard_eval_points_objectid_field = structure_hazard_eval_points.manager.properties["objectIdField"]
        structure_risk_plan_objectid_field = structure_risk_plan.manager.properties["objectIdField"]
        structure_hazard_monitoring_objectid_field = structure_hazard_monitoring.manager.properties["objectIdField"]
        structure_worksite_axis_objectid_field = structure_worksite_axis.manager.properties["objectIdField"]
        structure_grid_objectid_field = structure_grid.manager.properties["objectIdField"]
        structure_hazard_risk_objectid_field = structure_hazard_risk.manager.properties["objectIdField"]
        ### NOTE: ADD NEW LAYERS HERE ###

        sifs_archive_item = gis.content.get(sifs_archive_item_id)
        sifs_archive_waypoints = sifs_archive_item.layers[0]
        sifs_archive_worksites = sifs_archive_item.layers[1]
        sifs_archive_points = sifs_archive_item.layers[2]
        sifs_archive_tracklog = sifs_archive_item.layers[3]
        sifs_archive_segments = sifs_archive_item.layers[4]
        sifs_archive_divisions = sifs_archive_item.layers[5]
        sifs_archive_branches = sifs_archive_item.layers[6]
        sifs_archive_incident_areas = sifs_archive_item.layers[7]
        # Search Coverage
        search_coverage_archive_item = gis.content.get(search_coverage_archive_id)
        search_coverage_archive_points = search_coverage_archive_item.layers[0]

        # Structure hazards
        structures_archive_item = gis.content.get(structure_hazard_archive)

        structure_hazard_eval_archive = structures_archive_item.layers[0]
        structure_risk_plan_archive = structures_archive_item.layers[1]
        structure_hazard_monitoring_archive = structures_archive_item.layers[2]
        structure_worksite_axis_archive = structures_archive_item.layers[3]
        structure_grid_archive = structures_archive_item.layers[4]
        structure_hazard_risk_archive = structures_archive_item.layers[5]
        ### NOTE: ADD NEW LAYERS HERE ###

        # Get incident area geometry and start/end dates
        event_name_where_clause = "event_name = '{}'".format(incident_id)
        all_incident_areas = sifs_incident_areas.query(where=event_name_where_clause)
        incident_sr = sifs_incident_areas.properties.extent['spatialReference']
        incident_geom_dict = all_incident_areas.features[0].geometry
        incident_geom_dict['spatialReference'] = incident_sr
        incident_geom = Geometry(incident_geom_dict)
        incident_filter = intersects(incident_geom, sr=incident_sr)
        inc_start = all_incident_areas.features[0].attributes['incident_start_date']
        inc_end = all_incident_areas.features[0].attributes['incident_end_date']
        inc_start_date = dt.fromtimestamp(inc_start/1000).strftime("%m/%d/%Y 00:00:00")
        if inc_end:
            inc_end_date = dt.fromtimestamp(inc_end/1000).strftime("%m/%d/%Y 23:59:59")
        else:
            inc_end_date = dt.now().strftime("%m/%d/%Y 23:59:59")
        
        where_clause = "CreationDate > '{}' AND CreationDate < '{}' AND delete_this <> 'yes_delete'".format(inc_start_date, inc_end_date)
        search_coverage_where = "CreationDate > '{}' AND CreationDate < '{}' AND analysis_type <> 'Staging Run'".format(inc_start_date, inc_end_date)
        ### NOTE: ADD NEW LAYERS HERE ###  <= If the where clause needs to be different for net fields then you will need to create a new variable
       ############################################################################################################################################
        def backup_gdb():
            update_progressor(progressor_box, "Exporting Data to GDB...")
            incident_name = incident_id
            # local folder on your machine to save attachments and data
            local_folder_name = 'Incident_Archive'
            
            
            # Delete the local File Geodatabase with attachements and the zipped up file geodatabase. True will delete the file geodatabase and the zip file
            
            # Delete the raw attachment files. Files are downloaded in to a folder before being added to a file geodatabase. True will delete these folders, False will not
            clean_up_raw_attachments = True
            
            # Upload as item in organization. The new Item will be output near the end of the notebook. True will upload and False will not
            upload_to_org = True
    
            item = gis.content.get(sifs_item_id)
    
            def download_attachment(att_dict, folder, counter):
                '''
                download a single attachment
                
                Parameters:
                    att_dict: dict
                        Dictionary of attachment information
                    
                Returns:
                    None
                '''
                # unpack the dictionary
                feature_oid = att_dict['PARENTGLOBALID']
                att_name = att_dict['NAME']
                att_url = att_dict['DOWNLOAD_URL']
                att_oid = att_dict['ID']
            
                if counter % 1000 == 0:
                    print(f"\tDownloading {counter} - {counter + 999}")
                
                # create a folder for each feature
                guid_folder = os.path.join(folder, feature_oid)
                if not os.path.exists(guid_folder):
                    os.makedirs(guid_folder)
            
                # create a folder for each attachment
                att_folder = os.path.join(guid_folder, str(att_oid))
                if not os.path.exists(att_folder):
                    os.makedirs(att_folder)
            
                # get the attachment and write to disk
                att_path = os.path.join(att_folder, att_name)
                response = requests.get(att_url)
            
                with open(att_path, 'wb') as f:
                    f.write(response.content)
                return 
            
            
            def download_attachments(attachments, folder):
                '''
                Download all attachments from a list of attachments
                
                Parameters:
                    attachments: list
                        List of attachment dictionaries
                    folder: str
                        Path to the folder to save the attachments
                    
                Returns:
                    None
                '''
                print(f"Downloading {len(attachments)} attachments")
                df_attachments = pd.DataFrame(attachments)
                for glob in df_attachments.PARENTGLOBALID.unique():
                    guid_folder = os.path.join(folder, glob)
                    if not os.path.exists(guid_folder):
                        os.makedirs(guid_folder)
                with ThreadPoolExecutor() as executor:
                    futures = []
                    results = []
                    for i, att in enumerate(attachments):
                        futures.append(executor.submit(download_attachment, att, folder, i))
                    for f in as_completed(futures):
                        results.append(f.result())
                return
            
            
            def replica_no_attachments(item, folder):
                '''
                Download all attachments from a list of attachments
                
                Parameters:
                    item: arcgis.gis.Item
                        Item to export
                    layer_index: int
                        Index of the layer to export
                    folder: str
                        Path to the folder to save the attachments
                    
                Returns:
                    None
                '''
                ##############################
                layer_queries = {
                str(l.properties.id): {
                    "where": f"event_name = '{incident_name}' AND CreationDate > '{inc_start_date}' AND CreationDate < '{inc_end_date}' AND delete_this <> 'yes_delete'"
                }
                for l in item.layers + item.tables
                }
                ###################################
    
                Data_Layers = []
                for lyr in item.layers:
                    where_clause = f"event_name = '{incident_name}'"
                    count = lyr.query(where=where_clause, return_count_only=True)
                    if count > 0:
                        Data_Layers.append(lyr)
                
                gis = item._gis
                today = time.strftime('%Y%m%d %H:%M')
                flc = FeatureLayerCollection.fromitem(item)
                resp = gis._con.post(
                    flc.url + "/createReplica",
                    {
                        "layers": ",".join([str(l.properties.id) for l in Data_Layers + item.tables]),
                        "replicaName": f"{item.title}_{today}",
                        "returnAttachments": False,
                        "returnAttachmentsDatabyURL": False,
                        "syncModel": "none",
                        "dataFormat": "filegdb",
                        "attachmentsSyncDirection": "none",
                        "async": True,
                        "syncDirection": "download",
                        "layerQueries": layer_queries
                    }
                )
            
                while gis._con.get(resp['statusUrl'])['status'] in['Pending', 'ExportingData']:
                    time.sleep(5)
            
                    status_resp = gis._con.get(resp['statusUrl'])
            
                    if status_resp['status'] == 'Completed':
                        print("Replica created successfully")
                        result_url = status_resp['resultUrl']
            
                        result_file = gis._con.get(result_url, out_path=folder, file_name=f"{incident_name}_Incident_Data.zip")
                return result_file
    
            def zip_file_geodatabase(input_file_geodatabase, zip_file_path):
                '''
                Zip a file geodatabase
            
                Parameters:
                    input_file_geodatabase: str
                        Path to the file geodatabase
                    zip_file_path: str
                        Path to the output zip file
            
                Returns:
                    str
                        Path to the output zip file
                '''
                
                zip_folder = os.path.dirname(zip_file_path)
                publish_fgdb = os.path.join(zip_folder, os.path.basename(zip_file_path).replace('.zip','.gdb'))
                if os.path.exists(publish_fgdb):
                    shutil.rmtree(publish_fgdb)
                shutil.copytree(
                    input_file_geodatabase,
                    publish_fgdb,
                    ignore=shutil.ignore_patterns("*.lock")
                )
            
                with ZipFile(zip_file_path, "w") as zipper:
                    for root, dirs, files in os.walk(publish_fgdb):
                        for file in files:
                            fpath = os.path.join(root, file)
                            zpath = os.path.relpath(
                                        os.path.join(root, file),
                                        os.path.join(publish_fgdb, '..')
                                    )
                            zipper.write(
                                fpath,
                                zpath
                            )
                shutil.rmtree(publish_fgdb)
                return zip_file_path
            # Configure scratch and working directories
            root_dir = os.getcwd()
            # Create Directory for the zip file to download in to
            if root_dir == '/arcgis':
                working_dir = os.path.join(root_dir, "home",local_folder_name)
                if not os.path.exists(working_dir):
                    os.mkdir(working_dir)
            else:
                working_dir = os.path.join(root_dir,local_folder_name)
                if not os.path.exists(working_dir):
                    os.mkdir(working_dir)
            
            # create a folder specific to each item
            item_folder = os.path.join(working_dir, item.id)
            if not os.path.exists(item_folder):
                os.makedirs(item_folder)
    
            # get the current date to use in output file naming
            today = time.strftime("%Y%m%d_%H:%M")
            
            
            
            
            # generate an export of the data without attachments
            replica_zip = replica_no_attachments(item, item_folder)
            
            # extract the data
            with ZipFile(replica_zip, 'r') as zip_ref:
                zip_res = zip_ref.extractall(item_folder)
            
            # get the path for the recently extracted fgdb
            fgdb_name = [i for i in os.listdir(item_folder) if i.endswith('.gdb')][0]
            fgdb = os.path.join(item_folder, fgdb_name)
            print(fgdb)
    
            arcpy.env.workspace = fgdb
    
            datasets = arcpy.ListDatasets(feature_type='feature')
            datasets = [''] + datasets if datasets is not None else []
            
            for ds in datasets:
                for fc in arcpy.ListFeatureClasses(feature_dataset=ds):
                    path = os.path.join(arcpy.env.workspace, ds, fc)
                    print(path)
            # iterate through the item's layers
            for layer in item.layers:
                local_fc_name = layer.properties.name.replace(" ", "_") ## CHANGE
                print('Web Layer Name: ',layer.properties.name)
                #print('Local FC Name: ',local_fc_name) ## CHANGE
            
                # get a list of all attachments
                all_attachments = layer.attachments.search(f"event_name = '{incident_name}'")
                # all_attachments = layer.attachments.search()
                
                if len(all_attachments) > 0: ## CHANGE
                    # create a folder for the attachments
                    attachment_folder = os.path.join(item_folder, f'attachments_{layer.properties.id}')
                    if not os.path.exists(attachment_folder):
                        os.makedirs(attachment_folder)
                
                    # download the attachments - multithreaded so this could consume a lot of bandwidth
                    
                    download_attachments(all_attachments, attachment_folder)
                
                    # generate a join table to link the attachments to the features
                    df_attachments = pd.DataFrame(all_attachments)
                    df_attachments['att_path'] = df_attachments.apply(lambda x: os.path.join(attachment_folder, str(x['PARENTGLOBALID']), str(x['ID']), x['NAME']), axis=1)
                    df_attachments.PARENTGLOBALID = "{"+df_attachments.PARENTGLOBALID.str.upper()+"}"
                    join_table = os.path.join(item_folder, f'join_table{layer.properties.id}.csv')
                    df_attachments.to_csv(join_table, index=False)
                
                    # find the path to the feature class
                    fc = os.path.join(fgdb, local_fc_name) ## CHANGE (needed to account for spaces in a feature class name)
                
                    # enable attachments on the feature class
                    arcpy.management.EnableAttachments(fc)
                
                    # add the attachments to the feature class using the join table
                    arcpy.management.AddAttachments(
                        os.path.join(fgdb, local_fc_name), ## Change
                        'GlobalID', 
                        join_table, 
                        'PARENTGLOBALID', 
                        'att_path')
                    if clean_up_raw_attachments == True:
                        # remove the attachment folders
                        shutil.rmtree(attachment_folder)
            
            print(f'Data locally Stored here: {fgdb}')
            if upload_to_org == True:
                # zip the file geodatabase
                out_zip = zip_file_geodatabase(fgdb, os.path.join(item_folder, f"{incident_name}_Incident_Data.zip"))
                
                
                
                # add the zip file to arcgis online
                item_fgdb = gis.content.add(
                    item_properties={
                        'type': 'File Geodatabase',
                        'tags': f'{incident_name}, archive',
                        'snippet': f'Automated export of the {incident_name} incident database, exported on {today} UTC (-5 hours for Easten).',
                        'description': f'Automated export of the {incident_name} incident database, exported on {today} UTC (-5 hours for Easten). SIFS layer includes waypoints, tracklogs, event points, search segments, branches, divisions, and incident area. This export includes ALL layers, even if there is no data in said layer for the incident.',
                        'title': f'{incident_name} Incident Data Archive'
                    },
                    data=out_zip 
                )
                item_fgdb
            # delete the file geodatabase
            shutil.rmtree(fgdb, ignore_errors=True)
            # delete the file geodatabase
            shutil.rmtree(out_zip, ignore_errors=True)
            shutil.rmtree(working_dir, ignore_errors=True)
            
            # incident_data_url = f"https://www.arcgis.com/home/item.html?id={item_fgdb.id}"
    
            # # Share to SARGeo Group B
            # sargeoB = gis.groups.get('6ff2b44443f54e5da04a1100ff6c3bd7')
            # item_fgdb.share(org=False, groups=sargeoB)
    
            # Lock item so it doesn't get deleted
            item_fgdb.protect()

        if gdb_checkbox.value:
            backup_gdb()
        
        ################################################################################################################################################

        
        new_incident_areas_to_archive, new_incident_areas_ids = identify_features_to_archive(sifs_incident_areas,
                                                                                 sifs_archive_incident_areas,
                                                                                 incident_id=incident_id)
        
        #Identify features from incident that have not yet been moved to the archive feature service
        new_waypoints_to_archive, new_waypoints_ids = identify_incident_data(sifs_waypoints,
                                                                                sifs_archive_waypoints,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_worksites_to_archive, new_worksites_ids = identify_incident_data(sifs_worksites,
                                                                                sifs_archive_worksites,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_points_to_archive, new_points_ids = identify_incident_data(sifs_points,
                                                                                sifs_archive_points,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_tracklog_to_archive, new_tracklogs_ids = identify_incident_data(sifs_tracklog,
                                                                                sifs_archive_tracklog,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_segments_to_archive, new_segments_ids = identify_incident_data(sifs_segments,
                                                                                sifs_archive_segments,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_divisions_to_archive, new_divisions_ids = identify_incident_data(sifs_divisions,
                                                                                sifs_archive_divisions,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        new_branches_to_archive, new_branches_ids = identify_incident_data(sifs_branches,
                                                                                sifs_archive_branches,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        # Search Coverage
        time.sleep(3)
        new_search_coverage_to_archive, new_search_coverage_ids = identify_incident_data(search_coverage_points,
                                                                                search_coverage_archive_points,
                                                                                incident_id=incident_id,
                                                                                where_clause=search_coverage_where,
                                                                                incident_filter=incident_filter)

        # Structure Hazard layers
        new_structure_hazard_eval_points_to_archive, new_structure_hazard_eval_points_ids = identify_incident_data(structure_hazard_eval_points,
                                                                                structure_hazard_eval_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        
        new_structure_risk_plan_to_archive, new_structure_risk_plan_ids = identify_incident_data(structure_risk_plan,
                                                                                structure_risk_plan_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        
        new_structure_hazard_monitoring_to_archive, new_structure_hazard_monitoring_ids = identify_incident_data(structure_hazard_monitoring,
                                                                                structure_hazard_monitoring_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        
        new_structure_worksite_axis_to_archive, new_structure_worksite_axis_ids = identify_incident_data(structure_worksite_axis,
                                                                                structure_worksite_axis_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        
        new_structure_grid_to_archive, new_structure_grid_ids = identify_incident_data(structure_grid,
                                                                                structure_grid_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)

        new_structure_hazard_risk_to_archive, new_structure_hazard_risk_ids = identify_incident_data(structure_hazard_risk,
                                                                                structure_hazard_risk_archive,
                                                                                incident_id=incident_id,
                                                                                where_clause=where_clause,
                                                                                incident_filter=incident_filter)
        ### NOTE: ADD NEW LAYERS HERE ### 

        # Only archive new features if new features were found to archive
        if new_incident_areas_to_archive:
            update_progressor(progressor_box, "Archiving {} incident area(s) ...".format(str(len(new_incident_areas_to_archive))))
            incident_areas_results = send_edits_to_service(new_incident_areas_to_archive, sifs_archive_incident_areas)
            incident_areas_results_lookup = {result['globalId'] : result['objectId'] for result in incident_areas_results}
        else:
            update_progressor(progressor_box, "No incident areas to archive ...")
            
        if new_waypoints_to_archive:
            update_progressor(progressor_box, "Archiving {} waypoint(s) ...".format(str(len(new_waypoints_to_archive))))
            waypoints_results = send_edits_to_service(new_waypoints_to_archive, sifs_archive_waypoints, incident_id=incident_id)
            waypoint_results_lookup = {result['globalId'] : result['objectId'] for result in waypoints_results}
            # if new_waypoints_ids and waypoint_results_lookup:
            #     update_progressor(progressor_box, "Archiving attachments from waypoints features ...")
            #     assessments_attachments_folder = os.path.join("home", "waypoints")
            #     migrate_attachments(sifs_waypoints,sifs_archive_waypoints,new_waypoints_ids,waypoint_results_lookup,assessments_attachments_folder)
            # else:
            #     update_progressor(progressor_box, "No attachments from features to archive ...")
        else:
            update_progressor(progressor_box, "No waypoints to archive ...")
        if new_worksites_to_archive:
            update_progressor(progressor_box, "Archiving {} worksite(s) ...".format(str(len(new_worksites_to_archive))))
            worksites_results = send_edits_to_service(new_worksites_to_archive, sifs_archive_worksites, incident_id=incident_id)
            worksites_results_lookup = {result['globalId'] : result['objectId'] for result in worksites_results}
            '''if new_worksites_ids and worksites_results_lookup:
                update_progressor(progressor_box, "Archiving attachments from worksites features ...")
                assessments_attachments_folder = os.path.join("home", "worksites")
                migrate_attachments(sifs_worksites,sifs_archive_worksites,new_worksites_ids,worksites_results_lookup,assessments_attachments_folder)
            else:
                update_progressor(progressor_box, "No attachments from features to archive ...")'''
        else:
            update_progressor(progressor_box, "No worksite(s) to archive ...")
        if new_points_to_archive:
            update_progressor(progressor_box, "Archiving {} event point(s) ...".format(str(len(new_points_to_archive))))
            points_results = send_edits_to_service(new_points_to_archive, sifs_archive_points, incident_id=incident_id)
            points_results_lookup = {result['globalId'] : result['objectId'] for result in points_results}
            if new_points_ids and points_results_lookup:
                update_progressor(progressor_box, "Archiving attachments from points features ...")
                assessments_attachments_folder = os.path.join("home", "points")
                migrate_attachments(sifs_points,sifs_archive_points,new_points_ids,points_results_lookup,assessments_attachments_folder)
            else:
                update_progressor(progressor_box, "No attachments from features to archive ...")
        else:
            update_progressor(progressor_box, "No event points to archive ...")
        if new_tracklog_to_archive:
            update_progressor(progressor_box, "Archiving {} tracklog(s) ...".format(str(len(new_tracklog_to_archive))))
            tracklog_results = send_edits_to_service(new_tracklog_to_archive, sifs_archive_tracklog, incident_id=incident_id)
            tracklog_results_lookup = {result['globalId'] : result['objectId'] for result in tracklog_results}
            '''if new_tracklogs_ids and tracklog_results_lookup:
                update_progressor(progressor_box, "Archiving attachments from tracklogs features ...")
                assessments_attachments_folder = os.path.join("home", "tracklogs")
                migrate_attachments(sifs_tracklog,sifs_archive_tracklog,new_tracklogs_ids,tracklog_results_lookup,assessments_attachments_folder)
            else:
                update_progressor(progressor_box, "No attachments from features to archive ...")'''
        else:
            update_progressor(progressor_box, "No tracklogs to archive ...")
        if new_segments_to_archive:
            update_progressor(progressor_box, "Archiving {} search segment(s) ...".format(str(len(new_segments_to_archive))))
            segments_results = send_edits_to_service(new_segments_to_archive, sifs_archive_segments, incident_id=incident_id)
            segments_results_lookup = {result['globalId'] : result['objectId'] for result in segments_results}
            '''if new_segments_ids and segments_results_lookup:
                update_progressor(progressor_box, "Archiving attachments from segments features ...")
                assessments_attachments_folder = os.path.join("home", "segments")
                migrate_attachments(sifs_segments,sifs_archive_segments,new_segments_ids,segments_results_lookup,assessments_attachments_folder)
            else:
                update_progressor(progressor_box, "No attachments from features to archive ...")'''
        else:
            update_progressor(progressor_box, "No search segments to archive ...")
        if new_divisions_to_archive:
            update_progressor(progressor_box, "Archiving {} division(s) ...".format(str(len(new_divisions_to_archive))))
            divisions_results = send_edits_to_service(new_divisions_to_archive, sifs_archive_divisions, incident_id=incident_id)
            divisions_results_lookup = {result['globalId'] : result['objectId'] for result in divisions_results}
            '''if new_divisions_ids and divisions_results_lookup:
                update_progressor(progressor_box, "Archiving attachments from divisions features ...")
                assessments_attachments_folder = os.path.join("home", "divisions")
                migrate_attachments(sifs_divisions,sifs_archive_divisions,new_divisions_ids,divisions_results_lookup,assessments_attachments_folder)
            else:
                update_progressor(progressor_box, "No attachments from features to archive ...")'''
        else:
            update_progressor(progressor_box, "No divisions to archive ...")
        if new_branches_to_archive:
            update_progressor(progressor_box, "Archiving {} branch(es) ...".format(str(len(new_branches_to_archive))))
            branches_results = send_edits_to_service(new_branches_to_archive, sifs_archive_branches, incident_id=incident_id)
            branches_results_lookup = {result['globalId'] : result['objectId'] for result in branches_results}
            '''if new_branches_ids and branches_results_lookup:
                update_progressor(progressor_box, "Archiving attachments from branches features ...")
                assessments_attachments_folder = os.path.join("home", "branches")
                migrate_attachments(sifs_branches,sifs_archive_branches,new_branches_ids,branches_results_lookup,assessments_attachments_folder)
            else:
                update_progressor(progressor_box, "No attachments from features to archive ...")'''
        else:
            update_progressor(progressor_box, "No branches to archive ...")
        '''if new_incident_areas_ids and incident_areas_results_lookup:
            update_progressor(progressor_box, "Archiving attachments from assessment features ...")
            assessments_attachments_folder = os.path.join("home", "incident_areas")
            migrate_attachments(sifs_incident_areas,sifs_archive_incident_areas,new_incident_areas_ids,incident_areas_results_lookup,assessments_attachments_folder)
        else:
            update_progressor(progressor_box, "No attachments from incident areas to archive ...")'''
        
        # Search Coverage
        if new_search_coverage_to_archive:
            update_progressor(progressor_box, "Archiving {} search coverage ...".format(str(len(new_search_coverage_to_archive))))
            time.sleep(3)
            search_coverage_results = send_edits_to_service(new_search_coverage_to_archive, search_coverage_archive_points, incident_id=incident_id)
            search_coverage_results_lookup = {result['globalId'] : result['objectId'] for result in search_coverage_results}
            '''if new_search_coverage_ids and search_coverage_results_lookup:
                update_progressor(progressor_box, "Archiving attachments from search_coverage features ...")
                assessments_attachments_folder = os.path.join("home", "search_coverage")
                migrate_attachments(search_coverage_points,search_coverage_archive_points,new_search_coverage_ids,search_coverage_results_lookup,assessments_attachments_folder)
            else:
                update_progressor(progressor_box, "No attachments from features to archive ...")'''
        else:
            update_progressor(progressor_box, "No search coverage to archive ...")
            
        if new_structure_hazard_eval_points_to_archive:
            update_progressor(progressor_box, "Archiving {} Structure Eval data".format(str(len(new_structure_hazard_eval_points_to_archive))))
            hazard_eval_result = send_edits_to_service(new_structure_hazard_eval_points_to_archive, structure_hazard_eval_archive, incident_id=incident_id)
            eval_result_lookup = {result['globalId'] : result['objectId'] for result in hazard_eval_result}
        else:
            update_progressor(progressor_box, "No structure hazard evals to archive ...")

        
        if new_structure_risk_plan_to_archive:
            update_progressor(progressor_box, "Archiving {} Risk Plan data".format(str(len(new_structure_risk_plan_to_archive))))
            risk_plan_result = send_edits_to_service(new_structure_risk_plan_to_archive, structure_risk_plan_archive, incident_id=incident_id)
            risk_plan_lookup = {result['globalId'] : result['objectId'] for result in risk_plan_result}
        else:
            update_progressor(progressor_box, "No structure Risk Plan data to archive ...")

        
        if new_structure_hazard_monitoring_to_archive:
            update_progressor(progressor_box, "Archiving {} Structure Hazard Monitoring data".format(str(len(new_structure_hazard_monitoring_to_archive))))
            Structure_hazard_monitor_results = send_edits_to_service(new_structure_hazard_monitoring_to_archive, structure_hazard_monitoring_archive, incident_id=incident_id)
            Structure_hazard_monitor_lookup = {result['globalId'] : result['objectId'] for result in Structure_hazard_monitor_results}
        else:
            update_progressor(progressor_box, "No structure Hazard Moniroting data to archive ...")

        if new_structure_worksite_axis_to_archive:
            update_progressor(progressor_box, "Archiving {} Worksite Axis data".format(str(len(new_structure_worksite_axis_to_archive))))
            worksite_axis_results = send_edits_to_service(new_structure_worksite_axis_to_archive, structure_worksite_axis_archive, incident_id=incident_id)
            worksite_axis_lookup = {result['globalId'] : result['objectId'] for result in worksite_axis_results}

        else:
            update_progressor(progressor_box, "No structure Hazard Moniroting data to archive ...")

        if new_structure_grid_to_archive:
            update_progressor(progressor_box, "Archiving {} Structure Grid".format(str(len(new_structure_grid_to_archive))))
            structure_grid_results = send_edits_to_service(new_structure_grid_to_archive, structure_grid_archive, incident_id=incident_id)
            structure_grid_lookup = {result['globalId'] : result['objectId'] for result in structure_grid_results}

        else:
             update_progressor(progressor_box, "No structure Structure Grid data to archive ...")

        if new_structure_hazard_risk_to_archive:
            update_progressor(progressor_box, "Archiving {} Structure Hazard Risk data".format(str(len(new_structure_hazard_risk_to_archive))))
            hazard_risk_results = send_edits_to_service(new_structure_hazard_risk_to_archive, structure_hazard_risk_archive, incident_id=incident_id)
            hazard_risk_lookup = {result['globalId'] : result['objectId'] for result in hazard_risk_results}

        else:
            update_progressor(progressor_box, "No structure Hazard Risk data to archive ...")
        ### NOTE: ADD NEW LAYERS HERE ### 


        ''' 
        
        #only migrate attachments if there are new features to archive
        if new_features_ids and assessments_results_lookup:
            update_progressor(progressor_box, "Archiving attachments from assessment features ...")
            assessments_attachments_folder = os.path.join("home", "assessments")
            migrate_attachments(damage_assessments_layer,
                                damage_assessments_archive_layer,
                                new_features_ids,
                                assessments_results_lookup,
                                assessments_attachments_folder)
        else:
            update_progressor(progressor_box, "No attachments from features to archive ...")
        
        # Process Related Comments ******************************************
        
        if has_comments:
            new_comments_to_archive = []
            new_comments_ids = []
            comments_results_lookup = {}

            comments_table = damage_assessments_item.tables[0]
            comments_table_objectid_field = comments_table.manager.properties["objectIdField"]
            comments_archive_table = damage_assessments_archive_item.tables[0]

            update_progressor(progressor_box, "Identifying related comments to archive from incident: {}...".format(incident_id))
            oid_list = [f.get_value(damage_assessments_objectid_field) for f in all_incident_features.features]
            oid_list_string = ",".join([str(oid) for oid in oid_list])


            # search for related records based on the incident ID
            related_comments_results = damage_assessments_layer.query_related_records(object_ids=oid_list_string,
                                                                                      relationship_id=0,
                                                                                      out_fields=comments_table_objectid_field)
            related_comments_oids = get_related_records_oids(related_comments_results, comments_table_objectid_field)
            related_comments_oids_list_string = ",".join([str(oid) for oid in list(related_comments_oids.keys())])

            # Only look for new comments to archive if new comments were found in previous search
            if related_comments_oids_list_string:
                new_comments_to_archive, new_comments_ids = identify_features_to_archive(comments_table,
                                                                                         comments_archive_table,
                                                                                         object_ids=related_comments_oids_list_string)

            # Only archive new comments if they are not already in the archive feature service
            if new_comments_to_archive:
                update_progressor(progressor_box, "Archiving {} comment(s) ...".format(str(len(new_comments_to_archive))))
                comments_results = send_edits_to_service(new_comments_to_archive, comments_archive_table)
                comments_results_lookup = {result['globalId'] : result['objectId'] for result in comments_results}
            else:
                update_progressor(progressor_box, "No comments to archive ...")

            #only migrate attachments if there are new comments to archive
            if new_features_ids and comments_results_lookup:
                update_progressor(progressor_box, "Archiving attachments from related comments ...")
                comments_attachments_folder = os.path.join("home", "comments")
                migrate_attachments(comments_table,
                                    comments_archive_table,
                                    new_comments_ids,
                                    comments_results_lookup,
                                    comments_attachments_folder)
            else:
                update_progressor(progressor_box, "No attachments from related comments to archive ...")
        '''
        time.sleep(5)
        update_progressor(progressor_box, "Finished archiving incident: {} ...".format(incident_id))

        SIFS = gis.content.get(sifs_archive_item_id)
        incidentArea = SIFS.layers[7]
        incident_query = incidentArea.query(where=f"event_name = '{incident_id}'")
        incident_feature = incident_query.features[0]
        if fema_checkbox.value:
            incident_feature.attributes["FEMA_Deployed"] = "Yes"
        else:
            incident_feature.attributes["FEMA_Deployed"] = "No"
            
        if gdb_checkbox.value:
            gdb_search = gis.content.search(query=f'title:"{incident_id} Incident Data Archive"')
            gdb = gdb_search[0]
            incident_data_url = f"https://www.arcgis.com/home/item.html?id={gdb.id}"
            incident_feature.attributes["database_link"] = incident_data_url
        else:
            incident_feature.attributes["database_link"] = "No GDB backup created"

        incidentArea.edit_features(updates=[incident_feature])
        

        out_item = widgets.Output()
        out_item.append_display_data(sifs_archive_item)
        if new_structure_hazard_eval_points_to_archive:
            out_item.append_display_data(structures_archive_item)

        archive_form.children += (out_item,)
        
        refresh_form(selected_index = finish_index)
        
    except Exception as e:
        update_progressor(progressor_box, error_state=True)
        tb = traceback.format_exc()
        out.append_stderr("Archive Unsuccessful. Please try again.\n")
        out.append_stderr("Error:\n{}\n".format(str(e)))
        out.append_stderr(tb)
    finally:
        archive_button.disabled = False
        
refresh_form(new_form=True)
incident_form.children[0].children[2].on_click(archive_features)
incident_form.children[1].children[2].on_click(delete_archived_features)
#incident_form.children[2].children[2].on_click(archive_features)
#incident_form.children[3].children[2].on_click(delete_archived_features)

display(toggle_code_button)
display(incident_form)